In [1]:
USE {
    dependencies("org.mariadb.jdbc:mariadb-java-client:3.5.4")
}

In [3]:
%useLatestDescriptors
%use dataframe(1.0.0-dev-10213)

In [2]:
import org.jetbrains.kotlinx.dataframe.DataFrame
import org.jetbrains.kotlinx.dataframe.api.describe
import org.jetbrains.kotlinx.dataframe.api.print
import org.jetbrains.kotlinx.dataframe.io.DbConnectionConfig
import org.jetbrains.kotlinx.dataframe.io.readDataFrameSchema
import org.jetbrains.kotlinx.dataframe.io.readSqlTable
import org.jetbrains.kotlinx.dataframe.io.readDataFrame
import org.jetbrains.kotlinx.dataframe.schema.DataFrameSchema
import org.jetbrains.kotlinx.dataframe.examples.jdbc.*
import java.sql.DriverManager
import java.util.*


**The IMDB Database Exploration: printing schemas for all non-system tables**

To run these examples, you need to install the MariaDB database server and load the dump into a database. Read more in the [Readme.md](https://github.com/zaleslaw/KotlinDataFrame-SQL-Examples/blob/master/Readme.md).

In [4]:
val dbConfig = DbConnectionConfig(URL, USER_NAME, PASSWORD)

val dataschemas = DataFrameSchema.readAllSqlTables(dbConfig)

dataschemas.forEach {
    println("--- Schema for Table ${it.key} ---")
    println(it.value)
    println()
}

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory
--- Schema for Table actors ---
id: Int
first_name: String?
last_name: String?
gender: String?

--- Schema for Table directors ---
id: Int
first_name: String?
last_name: String?

--- Schema for Table directors_genres ---
director_id: Int
genre: String
prob: Float?

--- Schema for Table movies ---
id: Int
name: String?
year: Int?
rank: Float?

--- Schema for Table movies_directors ---
director_id: Int
movie_id: Int

--- Schema for Table movies_genres ---
movie_id: Int
genre: String

--- Schema for Table roles ---
actor_id: Int
movie_id: Int
role: String



**The IMDB Data Quick Exploration: printing 100 rows from each non-system table**

In [5]:
import org.jetbrains.kotlinx.dataframe.io.readAllSqlTables

val dfs = DataFrame.readAllSqlTables(dbConfig, limit = 100).values

dfs.forEach {
    it.describe().print()
    it.print(5)
}

         name   type count unique nulls     top freq  mean       std         min     p25  median     p75         max
 0         id    Int   100    100     0       2    1 53,37 30,245679           2      26      54      80         106
 1 first_name String   100     93     0 Antonio    3  null      null       Ahmed  Equipe Krishna   Pauli Yussuf Abed
 2  last_name String   100     81     0      A.    5  null      null 'Chincheta' 't Hoen      A. Aagaard    a'Hiller
 3     gender String   100      1     0       M  100  null      null           M       M       M       M           M

   id first_name          last_name gender
 0  2    Michael 'babeepower' Viera      M
 1  3       Eloy        'Chincheta'      M
 2  4   Dieguito        'El Cigala'      M
 3  5    Antonio   'El de Chipiona'      M
 4  6       José       'El Francés'      M
...

         name   type count unique nulls     top freq  mean       std  min   p25    median     p75      max
 0         id    Int   100    100     0     

**Convert 10000 rows from _actors_ table to the dataframe**

In [6]:
val actorDf = DataFrame.readSqlTable(dbConfig, "actors", 10000)
actorDf

id,first_name,last_name,gender
2,Michael,'babeepower' Viera,M
3,Eloy,'Chincheta',M
4,Dieguito,'El Cigala',M
5,Antonio,'El de Chipiona',M
6,José,'El Francés',M
7,Félix,'El Gato',M
8,Marcial,'El Jalisco',M
9,José,'El Morito',M
10,Francisco,'El Niño de la Manola',M
11,Víctor,'El Payaso',M


**Find top-20 the most popular actor names**

In [7]:
import org.jetbrains.kotlinx.dataframe.api.groupBy

val top20ManActorNames = actorDf
    .groupBy { first_name }
    .count()
    .sortByDesc("count")
    .take(20)
    
top20ManActorNames

first_name,count
David,80
John,74
Daniel,49
Peter,46
Robert,46
Michael,45
Antonio,42
Luis,41
Paul,40
José,39


In [8]:
val top20ManActorNamesPlot = top20ManActorNames.plot {
    bars {
         x(first_name) {
            scale = categorical()
        }
        y(count)
    }
 }
 
 top20ManActorNamesPlot

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="byfL1C" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("byfL1C");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"count":[80.0,74.0,49.0,46.0,46.0,45.0,42.0,41.0,40.0,39.0,38.0,36.0,35.0,34.0,34.0,33.0,33.0,33.0,32.0,30.0],
"first_name":["David","John","Daniel","Peter","Robert","Michael","Antonio","Luis","Paul","José","Richard","A.","Carlos","Frank","Joe","Steve","Jorge","Roberto","Tony","James"]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"first_name",
"y":"count"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"first_name"
},{
"type":"int",
"column":"count"
}]
},
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 David 
 
 
 
 
 
 
 
 
 John 
 
 
 
 
 
 
 
 
 Daniel 
 
 
 
 
 
 
 
 
 Peter 
 
 
 
 
 
 
 
 
 Robert 
 
 
 
 
 
 
 
 
 Michael 
 
 
 
 
 
 
 
 
 Antonio 
 
 
 
 
 
 
 
 
 Luis 
 
 
 
 
 
 
 
 
 Paul 
 
 
 
 
 
 
 
 
 José 
 
 
 
 
 
 
 
 
 Richard 
 
 
 
 
 
 
 
 
 A. 
 
 
 
 
 
 
 
 
 Carlos 
 
 
 
 
 
 
 
 
 Frank 
 
 
 
 
 
 
 
 
 Joe 
 
 
 
 
 
 
 
 
 Steve 
 
 
 
 
 
 
 
 
 Jorge 
 
 
 
 
 
 
 
 
 Roberto 
 
 
 
 
 
 
 
 
 Tony 
 
 
 
 
 
 
 
 
 James 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 first_name

**Exploring the movies: trending rank by year depends on genre**

In [9]:
val sqlQuery = """
SELECT name, year, rank, genre
FROM movies
INNER JOIN movies_genres ON movies.id = movies_genres.movie_id
WHERE movies.rank > 0.0 and (movies_genres.genre = "Comedy" OR movies_genres.genre = "Horror")
"""

val ratedMoviesDf = DataFrame.readSqlQuery(dbConfig, sqlQuery)
ratedMoviesDf

name,year,rank,genre
$,1971,"6,400000",Comedy
"$1,000,000 Duck",1971,"5,000000",Comedy
$1000 a Touchdown,1939,"6,700000",Comedy
$30,1999,"7,500000",Comedy
"$40,000",1996,"9,600000",Comedy
"'?' Motorist, The",1906,"6,800000",Comedy
'A' gai waak,1983,"7,200000",Comedy
'A' gai waak juk jaap,1987,"7,200000",Comedy
'Babicky dobjejte presne!',1983,"5,600000",Horror
"'burbs, The",1989,"5,900000",Comedy


In [10]:
val olapCube = ratedMoviesDf
    .select { year and genre and rank }
    .groupBy { year and genre }
    .mean { rank named "meanRank" } 
    //.aggregate {  mean { rank } into "meanRank" } <-- alternative to the previous row
    .sortBy { year and genre }
    
olapCube

year,genre,meanRank
1892,Comedy,"5,100000"
1895,Comedy,"6,850000"
1896,Comedy,"3,550000"
1896,Horror,"4,250000"
1897,Comedy,"4,500000"
1898,Comedy,"4,620000"
1898,Horror,"3,200000"
1899,Comedy,"4,280000"
1899,Horror,"3,300000"
1900,Comedy,"4,612500"


In [11]:
val genreRankTrends = olapCube.plot {
    line {
        x(year)
        y(meanRank)
        color(genre)
        width = 1.0
    }
}
genreRankTrends

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="L9O9q1" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("L9O9q1");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"meanRank":[5.099999904632568,6.849999904632568,3.549999952316284,4.25,4.5,4.619999980926513,3.200000047683716,4.2799999713897705,3.299999952316284,4.612500011920929,4.7416666348775225,4.366666714350383,5.375000059604645,4.953846142842219,4.563636411320079,6.066666762034099,5.480000019073486,5.416666666666667,5.066666642824809,5.099999904632568,4.9666666984558105,5.900000095367432,6.424999952316284,5.840000057220459,4.900000095367432,6.041666666666667,7.300000190734863,6.061363621191545,6.300000190734863,5.734285681588309,7.400000095367432,6.079411752083722,6.754166642824809,6.596153809474065,7.044444455040826,7.699999809265137,6.724999934434891,6.650000095367432,6.954166650772095,7.136363679712469,7.766666730244954,6.607692351708045,7.5,6.35357141494751,7.400000095367432,6.830952360516503,6.849999904632568,6.936000027656555,7.3999998569488525,6.997435869314732,7.5,7.177777841356066,6.900000095367432,6.722727284286961,7.799999952316284,6.022222233197046,6.099999904632568,6.296732025208816,7.237499952316284,6.630882389405194,6.691666642824809,6.5793333339691165,6.580000019073486,6.450270290632505,5.485714163099017,6.380628283735345,6.291666746139526,6.475576031592585,5.785714285714286,6.371493214395791,7.349999904632568,6.318139544198679,6.5333333015441895,6.419000020027161,5.959999990463257,6.305479474263649,5.49333332379659,6.229752054884414,5.930769223433274,6.478787883363589,5.100000030854169,6.498170747989562,5.793333307902018,6.465789482781761,5.595833321412404,6.350406516858233,5.911764705882353,6.444202901660532,5.064285670007978,6.376190472622307,4.299999952316284,6.506896549257739,5.2399999618530275,6.461827948529233,5.875,6.472560968340897,6.424064169593036,6.012500047683716,6.394358982183994,5.342857190540859,6.305235599348058,6.033333407508002,6.354497360804724,5.925000071525574,6.310919530090245,5.529999995231629,6.048520713162845,4.915789491251895,6.075925922688143,4.254999974370003,6.076923084564698,4.500000009934108,6.16163524441749,4.713513528978503,6.166901384440946,5.736111130979326,5.967999986012777,4.958620700342902,6.043930614614762,4.674468101339137,6.061494247666721,5.15128205372737,6.068604659202487,4.899999987247378,5.820707074921541,4.341860485631366,5.913440869059614,4.591111132833693,5.625870638818883,4.474999970859951,5.777325577514116,4.936585356549519,5.450289022715794,4.371428574834551,5.642458088571133,4.398148123864774,5.33333333535383,4.526136344129389,5.439181283203482,4.282857121740069,5.432738091974032,4.788541663438082,5.575510189646766,4.477777787380749,5.6550295839648275,4.4319999885559085,5.471005929292306,4.6951612907071265,5.638461539025843,4.62166668176651,5.656497176084141,4.714285712156977,5.250276238878787,4.431372537332423,5.363963959990321,4.56301367119567,5.495049513212525,4.445882370892693,5.4574162811753855,4.3530302716024,5.53037382341991,4.347826071407484,5.544642858207226,4.27200001001358,5.6179389234717565,4.413333322207133,5.3643153559617485,4.598837209302325,5.288732401921716,4.307317069875515,5.331159419771554,4.407092195030645,5.45

**How are popular different genres in different years?**

In [12]:
val sqlQuery = """
SELECT name, year, genre
FROM movies
INNER JOIN movies_genres ON movies.id = movies_genres.movie_id
WHERE movies.year < 1940
"""

// The example of extension function usage
val moviesDf = dbConfig.readDataFrame(sqlQuery)
moviesDf

name,year,genre
"$1,000 Reward",1923,Western
"$10,000 Under a Pillow",1921,Animation
"$10,000 Under a Pillow",1921,Comedy
"$10,000 Under a Pillow",1921,Short
"$100,000",1915,Drama
$1000 a Touchdown,1939,Comedy
"$20,000 Carat, The",1913,Crime
"$20,000 Carat, The",1913,Drama
"$20,000 Carat, The",1913,Short
"$2500 Bride, The",1912,Drama


In [13]:
val aggregatedDF = moviesDf
    .select { year and genre }
    .groupBy { year and genre }
    .aggregate {  count()  into "count" }
    .sortBy { year and genre }
    
aggregatedDF

year,genre,count
1888,Short,2
1890,Short,3
1891,Short,2
1892,Animation,3
1892,Comedy,1
1892,Documentary,1
1892,Romance,1
1892,Short,4
1893,Short,1
1894,Comedy,1


In [15]:
val genreRankTrends = aggregatedDF.plot {
    points {
        x(year) // auto-generated df columns
        y(genre) {
            scale = categorical()
        }
        symbol = Symbol.CIRCLE_FILLED
        color = Color.BLUE
        fillColor(genre)
        size(count) {
            scale = continuous(1.0..30.0)
        }
    }
    layout.size = 1000 to 600
}

genreRankTrends

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="zAOvxl" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("zAOvxl");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"year":[1888.0,1890.0,1891.0,1892.0,1892.0,1892.0,1892.0,1892.0,1893.0,1894.0,1894.0,1894.0,1894.0,1895.0,1895.0,1895.0,1895.0,1895.0,1895.0,1896.0,1896.0,1896.0,1896.0,1896.0,1896.0,1897.0,1897.0,1897.0,1897.0,1897.0,1897.0,1897.0,1897.0,1898.0,1898.0,1898.0,1898.0,1898.0,1898.0,1898.0,1898.0,1898.0,1898.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1899.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1900.0,1901.0,1901.0,1901.0,1901.0,1901.0,1901.0,1901.0,1901.0,1901.0,1901.0,1901.0,1902.0,1902.0,1902.0,1902.0,1902.0,1902.0,1902.0,1902.0,1902.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1903.0,1904.0,1904.0,1904.0,1904.0,1904.0,1904.0,1904.0,1904.0,1904.0,1904.0,1904.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1905.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1906.0,1907.0,1907.0,1907.0,1907.0,1907.0,1907.0,1907.0,1907.0,1907.0,1907.0,1907.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1908.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1909.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1910.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1911.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1912.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1913.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1914.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1915.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1917.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1918.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1919.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1920.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1921.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1922.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1923.0,1924.0,1924.0,1924.0,

In [16]:
genreRankTrends.save("exported_plot.png")

C:\Users\Aleksey.Zinovev\IdeaProjects\KotlinDataFrame-SQL-Examples\notebooks\lets-plot-images\exported_plot.png

**Extracting data from the ResultSet with extension functions**

In [17]:
import org.jetbrains.kotlinx.dataframe.io.db.MariaDb

val props = Properties()
props.setProperty("user", USER_NAME)
props.setProperty("password", PASSWORD)

val TARANTINO_FILMS_SQL_QUERY = """
    SELECT name, year, rank,
    GROUP_CONCAT (genre) as "genres"
    FROM movies JOIN movies_directors ON movie_id = movies.id
    JOIN directors ON directors.id=director_id LEFT JOIN movies_genres ON movies.id = movies_genres.movie_id
    WHERE directors.first_name = "Quentin" AND directors.last_name = "Tarantino"
    GROUP BY name, year, rank
    ORDER BY year
    """

var dfTarantinoFilmsRs: DataFrame<*>

DriverManager.getConnection(URL, props).use { connection ->
    connection.createStatement().use { st ->
        st.executeQuery(TARANTINO_FILMS_SQL_QUERY).use { rs ->
            val dfTarantinoFilmsSchema = rs.readDataFrameSchema(MariaDb)
            dfTarantinoFilmsSchema.print()
            
            dfTarantinoFilmsRs = rs.readDataFrame(MariaDb)
            dfTarantinoFilmsRs
        }
    }
}

name: String?
year: Int?
rank: Float?
genres: String?


name,year,rank,genres
My Best Friend's Birthday,1987,"3,900000","Comedy,Drama"
Reservoir Dogs,1992,"8,300000","Crime,Mystery,Action,Thriller"
"""ER""",1994,null,null
Pulp Fiction,1994,"8,700000","Drama,Crime"
Four Rooms,1995,"5,900000","Comedy,Drama"
Jackie Brown,1997,"7,500000","Crime,Drama,Thriller"
"""Jimmy Kimmel Live""",2003,null,null
Kill Bill: Vol. 1,2003,"8,400000","Thriller,Action,Crime"
Kill Bill: Vol. 2,2004,"8,200000","Romance,Action,Thriller,Drama"
Inglorious Bastards,2006,null,"Drama,War,Action"


In [18]:
val df = dfTarantinoFilmsRs
    .fillNA { year }
    .with { 0 }
    .convert { year }.toInt()


In [19]:
df.filter { year > 2000 }

name,year,rank,genres
"""Jimmy Kimmel Live""",2003,null,null
Kill Bill: Vol. 1,2003,"8,400000","Thriller,Action,Crime"
Kill Bill: Vol. 2,2004,"8,200000","Romance,Action,Thriller,Drama"
Inglorious Bastards,2006,null,"Drama,War,Action"
